This notebook is used to use KIM to perform inverse modeling for the Advanced Terrestrial Simulator (ATS).


In [1]:
# Libraries
from pathlib import Path
import pandas as pd
import numpy as np

from kim.map import KIM
from kim.data import Data
from kim.mapping_model import MLP

import os
import jax

%load_ext autoreload
%autoreload 2


# Read the data
- The `Output_para.csv` file includes the ensemble of the seven parameters to be estimated, represented by $\mathbf{Y}$.
- The `Input_logq.csv` file includes the ensemble of the streamflow simulations generated by ATS, represented by $\mathbf{X}$.

In [2]:
# File and folder paths
f_para = Path("./data/Output_para.csv")
f_state = Path("./data/Input_logq.csv")


In [3]:
df_para, df_state = pd.read_csv(f_para, index_col=0),pd.read_csv(f_state, index_col=0)

In [4]:
y_keys, x_keys = df_para.keys(), df_state.keys()
y, x = df_para.values, df_state.values
x.shape, y.shape

((396, 1461), (396, 8))

# Configurations

## Preliminary analysis configuration

In [5]:
seed_shuffle = 1234
f_data_save = Path("./results/data")


In [6]:
# Data configuration
data_params = {
    "xscaler_type": "minmax",
    "yscaler_type": "minmax",
}

# Sensitivity analysis configuration
sensitivity_params = {
    "method": "pc", "metric": "it-knn",
    "sst": True, "ntest": 100, "alpha": 0.05, "k": 3,
    "n_jobs": min(os.cpu_count(), 100), "seed_shuffle": seed_shuffle,  # cap to available CPU cores; raise for HPC nodes
    "verbose": 1
}


## Ensemble learning configuration

In [7]:
Ns_train = 300
Ns_val = 50
hidden_activation = 'sigmoid'
final_activation = 'leaky_relu'
seed_ens = 1024
seed_predict = 3636
seed_dl = 10
seed_model = 100
training_verbose = 1
n_models = 100
n_jobs = min(os.cpu_count(), 20)  # cap to available CPU cores; raise for HPC nodes

f_kim_save1 = Path("./results/map_many2many")
f_kim_save2 = Path("./results/map_many2one")
f_kim_save3 = Path("./results/map_many2one_cond")


In [8]:
# Mapping parameters for each test below
map_configs = {
    "model_type": MLP,
    'n_model': n_models,
    'ensemble_type': 'ens_random',
    'model_hp_choices': {
        "depth": [1,3,5,6],
        "width_size": [3,6,10]
    },
    'model_hp_fixed': {
        "hidden_activation": hidden_activation,
        "final_activation": final_activation,
        "model_seed": seed_model
    },
    'optax_hp_choices': {
        'learning_rate': [0.01, 0.005, 0.003],
    },
    'optax_hp_fixed': {
        'nsteps': 300,
        'optimizer_type': 'adam',
    },
    'dl_hp_choices': {
    },
    'dl_hp_fixed': {
        'dl_seed': seed_dl,
        'num_train_sample': Ns_train,
        'num_val_sample': Ns_val,
        'batch_size': 64
    },
    'ens_seed': seed_ens,
    'training_parallel': True,
    'parallel_config': {
        'n_jobs': n_jobs, 
        'backend': 'loky',
        'verbose': 1
    },
    'device': None,
}

# Exploratory data analysis

**Expected runtime:** about 5 minutes on a 16-core laptop for the 1461 inputs x 8 outputs with `ntest=100` shuffles (it scales with the number of cores given by `n_jobs`). The ensemble training further below takes another ~7 minutes. The results are saved to `f_data_save` at the end of this notebook, so on a repeated run you can skip the analysis by uncommenting the loading cell below and skipping the `calculate_sensitivity` cell.

In [9]:
# # Load the exploratory analysis result if available
# data = Data(x, y)
# data.load(f_data_save)

In [10]:
# Perform the sensitivity analysis if not done
data = Data(x, y, **data_params, Ns_train=Ns_train, Ns_val=Ns_val)
data.calculate_sensitivity(**sensitivity_params)


Using the kNN-based information theoretic metrics ...
Performing pairwise analysis to remove insensitive inputs ...


  0%|          | 0/1461 [00:00<?, ?it/s]

  2%|▏         | 28/1461 [00:00<00:10, 131.16it/s]

  4%|▍         | 56/1461 [00:08<04:21,  5.38it/s] 

  6%|▌         | 84/1461 [00:11<03:19,  6.90it/s]

...

 96%|█████████▌| 1400/1461 [00:41<00:01, 44.34it/s]

 98%|█████████▊| 1428/1461 [00:42<00:00, 44.03it/s]

100%|█████████▉| 1456/1461 [00:42<00:00, 44.59it/s]

100%|██████████| 1461/1461 [00:42<00:00, 34.24it/s]

Performing conditional independence testing to remove redundant inputs ...


In [11]:
# data.sensitivity_mask
data.cond_sensitivity_mask

array([[False, False, False, ..., False,  True, False],
       [False, False, False, ..., False,  True, False],
       [False, False, False, ..., False,  True, False],
       ...,
       [False, False, False, ..., False,  True, False],
       [False, False, False, ..., False,  True, False],
       [False,  True, False, ..., False,  True, False]], shape=(1461, 8))

# Train the inverse mapping

Now, let's train the inverse mappings via ensemble learning. We are training three types of inverse mappings:
- `kim1`: The naive inverse mapping from all $\mathbf{Y}$ to all $\mathbf{X}$
- `kim2`: The knowledge-informed inverse mapping from sensitive $\mathbf{Y}$ to each of $\mathbf{X}$ using global sensitivity analysis
- `kim3`: The knowledge-informed inverse mapping from sensitive $\mathbf{Y}$ to each of $\mathbf{X}$ using global sensitivity analysis + redundancy filtering check


In [12]:
# Initialize three diffferent KIMs
kim1 = KIM(data, map_configs, map_option='many2many')
kim2 = KIM(data, map_configs, mask_option="sensitivity", map_option='many2one')
kim3 = KIM(data, map_configs, mask_option="cond_sensitivity", map_option='many2one')

# Train the mappings
kim1.train()
kim2.train()
kim3.train()



 Performing ensemble training in parallel with 100 model configurations...



[Parallel(n_jobs=20)]: Using backend LokyBackend with 20 concurrent workers.


  0%|          | 0/300 [00:00<?, ?it/s]

  0%|          | 1/300 [00:00<01:45,  2.84it/s]

  0%|          | 0/300 [00:00<?, ?it/s]

...

 51%|█████     | 153/300 [00:00<00:00, 245.97it/s]

 70%|███████   | 211/300 [00:01<00:00, 266.22it/s]

100%|██████████| 300/300 [00:01<00:00, 218.19it/s]


Training completes.


[Parallel(n_jobs=20)]: Done 100 out of 100 | elapsed:   18.0s finished


# Save all the results to disk

In [13]:
# Preliminary analysis results
data.save(f_data_save)

# Inverse mapping results
kim1.save(f_kim_save1)
kim2.save(f_kim_save2)
kim3.save(f_kim_save3)
